> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 5 · Notebook 03 — Candlesticks, and does the pattern work?

**Sessions:** S4 (Candlestick patterns) · S5 (Edge testing) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Compute candle anatomy safely, flat bars included.
2. Write the bullish engulfing rule and see why a hammer needs trend context.
3. Measure forward returns the honest way: entry at the next open.
4. Test 600 patterns at once and control false discoveries with Benjamini–Hochberg.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()
from scipy import stats

## 1. Candle anatomy

Every candlestick rule is built from five numbers per bar: the **body** `|C − O|`, the **range** `H − L`, the **upper shadow** `H − max(O, C)`, the **lower shadow** `min(O, C) − L`, and **body %** `body / range`. A flat bar (H = L, common in thin markets and pre-open data) has range 0: body % must be 0 there, not `inf` or NaN. `np.divide(a, b, out=np.zeros_like(a), where=b > 0)` does that.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def anatomy(o, h, l, c):
    body, rng_ = np.abs(c - o), h - l
    return {"body": body, "range": rng_,
            "upper": h - np.maximum(o, c),
            "lower": np.minimum(o, c) - l,
            "body_pct": np.divide(body, rng_, out=np.zeros_like(body), where=rng_ > 0)}

df = p.synthetic_ohlcv(1500, seed=7)
o, h, l, c, v = p.arrays(df)
o2, h2, l2, c2 = (np.append(a, 50.0) for a in (o, h, l, c))       # one flat bar at the end
mine = anatomy(o2, h2, l2, c2)
mine = p.check("anatomy", mine, p.anatomy(o2, h2, l2, c2))
pd.DataFrame(mine).tail(3)

## 2. Bullish engulfing

Bar `t−1` is bearish (`C < O`), bar `t` is bullish, and bar `t`'s body **covers** the previous body: `O_t <= C_{t−1}` and `C_t >= O_{t−1}`, with a strictly larger body. Output convention: a boolean per bar, `False` at bar 0.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bullish_engulfing(o, h, l, c):
    out = np.zeros(c.shape, dtype=bool)
    po, pc = o[:-1], c[:-1]                       # previous bar
    co, cc = o[1:], c[1:]                         # current bar
    out[1:] = (pc < po) & (cc > co) & (co <= pc) & (cc >= po) & ((cc - co) > (po - pc))
    return out

mine = p.attempt(bullish_engulfing, o, h, l, c)
mine = p.check("bullish_engulfing", mine, p.bullish_engulfing(o, h, l, c))
print(f"{int(np.sum(mine))} signals in {len(c)} bars")

**Context matters.** A hammer (long lower shadow, tiny upper shadow) is a reversal pattern only **after a decline**. The same shape after a rise is called a *hanging man* and means something else. Two hand-made sequences:

In [ ]:
def seq(trend):
    base = 100 + trend * np.arange(12)
    o_ = base + 0.1; c_ = base - 0.1 if trend < 0 else base + 0.3
    h_, l_ = np.maximum(o_, c_) + 0.2, np.minimum(o_, c_) - 0.2
    o_[-1], c_[-1] = base[-1] + 0.05, base[-1] + 0.25; h_[-1] = c_[-1] + 0.01; l_[-1] = o_[-1] - 1.5   # the hammer shape
    return o_, h_, l_, c_

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (name, trend) in zip(axes, [("after a decline", -0.5), ("after a rise", 0.5)]):
    o_, h_, l_, c_ = seq(trend)
    for i in range(12):
        col = p.PALETTE[2] if c_[i] >= o_[i] else p.PALETTE[7]
        ax.vlines(i, l_[i], h_[i], color=col, lw=1); ax.vlines(i, min(o_[i], c_[i]), max(o_[i], c_[i]), color=col, lw=6)
    flagged = p.hammer(o_, h_, l_, c_)[-1]
    ax.set(title=f"{name}: hammer → {flagged}", xticks=[])
plt.tight_layout(); plt.show()

## 3. Forward returns, honestly

A pattern is known only when bar `t` **closes**. The earliest fill is the **next bar's open**. The return of a trade held `h` bars is `O[t+1+h] / O[t+1] − 1`; where the exit is beyond the data, NaN.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def forward_returns(open_, horizon):
    n = open_.size
    out = np.full(open_.shape, np.nan)
    out[: n - horizon - 1] = open_[1 + horizon:] / open_[1:n - horizon] - 1
    return out

mine = p.attempt(forward_returns, o, 5)
mine = p.check("forward_returns", mine, p.forward_returns(o, 5))
mine[-8:]

In [ ]:
fwd = p.forward_returns(o, 5)
eng = p.bullish_engulfing(o, h, l, c)
mean_after, pval = p.permutation_pvalue(eng, fwd, n_perm=2000)
print(f"engulfing: {eng.sum()} signals, mean 5-bar return after {mean_after:+.3%} vs {np.nanmean(fwd):+.3%} unconditional")
print(f"permutation p-value {pval:.3f}: on this synthetic symbol, with no planted edge, nothing to see")

## 4. 600 tests, and the ones that look significant

Now the trap. Take 20 symbols and 30 patterns that are pure coin flips (each bar signals with probability 5%). Add one pattern that **cheats**: it signals when the next two opens rise, which it can't know at the close. Test each (pattern, symbol) pair with a one-sided t-test of the forward returns after the signal against the rest. At 5%, about 30 of the 600 honest tests pass by luck.

In [ ]:
uni = p.universe(20, 1000)
rng = np.random.default_rng(42)
rows = []
for sym, d in uni.items():
    o_, h_, l_, c_, _ = p.arrays(d)
    f = p.forward_returns(o_, 5)
    patterns = {f"coin{j:02d}": rng.random(len(c_)) < 0.05 for j in range(30)}
    patterns["cheat"] = np.r_[o_[2:] > o_[1:-1] * 1.004, [False, False]]
    for name, sig in patterns.items():
        ok = ~np.isnan(f)
        a, b = f[sig & ok], f[~sig & ok]
        rows.append({"pattern": name, "symbol": sym, "n": len(a), "mean": a.mean(),
                     "p": stats.ttest_ind(a, b, equal_var=False, alternative="greater").pvalue})
tests = pd.DataFrame(rows)
honest = tests["pattern"] != "cheat"
print(f"honest tests with p < 0.05: {(tests.loc[honest, 'p'] < 0.05).sum()} of {honest.sum()}")

**Benjamini–Hochberg** controls the *false discovery rate*: among what you call discoveries, the expected share of flukes stays below `α`. Sort the p-values; the adjusted value of the `i`-th smallest is `p_(i) · m / i`; then make them non-decreasing from the top (a running minimum from the largest down) and cap at 1. Return them in the **input order**.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bh_adjust(p_values):
    pv = np.asarray(p_values, dtype=float)
    m = pv.size
    order = np.argsort(pv)
    q = pv[order] * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]      # non-decreasing from the top
    out = np.empty(m)
    out[order] = np.minimum(q, 1.0)
    return out

mine = p.attempt(bh_adjust, tests["p"].to_numpy())
mine = p.check("bh_adjust", mine, p.bh_adjust(tests["p"].to_numpy()))
tests["q"] = mine
print(f"discoveries at FDR 5%: honest {(tests.loc[honest, 'q'] < 0.05).sum()}, cheat {(tests.loc[~honest, 'q'] < 0.05).sum()} of 20")

In [ ]:
fig, ax = plt.subplots()
ax.hist(tests.loc[honest, "p"], bins=20, alpha=0.8, label="30 coin-flip patterns × 20 symbols")
ax.hist(tests.loc[~honest, "p"], bins=20, alpha=0.8, label="the cheating pattern")
ax.axvline(0.05, color=p.PALETTE[7], ls="--", lw=1)
ax.set(xlabel="p-value", ylabel="tests", title="Honest p-values are uniform; a leak piles up at 0"); ax.legend(); plt.show()

BH removes the lucky coin flips and keeps the cheat. That's the other lesson: a pattern that survives every statistical test can still be a bug. **A result that looks too good is a look-ahead until proven otherwise.**

## Wrap-up

* Normalize candle rules and guard the flat bar; add trend context.
* Enter at the next open; compare with the unconditional distribution.
* Correct for the number of tests you ran (BH), then confirm out of sample.
* Graded versions: `labs/part05/week17_core` (patterns), `week18_groups` (`pattern_edge`, `bh_adjust`), Clinic W2 edge study.